In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, matthews_corrcoef,
                             average_precision_score, roc_auc_score,
                             classification_report)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train_final.csv')
test  = pd.read_csv('../data/test_final.csv')

X_train = train.drop(columns=['isFraud']).values
y_train = train['isFraud'].values
X_test  = test.drop(columns=['isFraud']).values
y_test  = test['isFraud'].values

print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")
print(f"GPU available: {torch.cuda.is_available()}")

Train: (15624, 192)
Test:  (2000, 192)
GPU available: False


In [2]:
# LSTM cần data được chuẩn hóa về cùng scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# LSTM cần input shape: (samples, timesteps, features)
# Mỗi giao dịch = 1 timestep, features = số cột
X_train_3d = X_train_sc.reshape(X_train_sc.shape[0], 1, X_train_sc.shape[1])
X_test_3d  = X_test_sc.reshape(X_test_sc.shape[0], 1, X_test_sc.shape[1])

print(f"Train 3D shape: {X_train_3d.shape}")
print(f"Test 3D shape:  {X_test_3d.shape}")

Train 3D shape: (15624, 1, 192)
Test 3D shape:  (2000, 1, 192)


In [3]:
class FraudDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = FraudDataset(X_train_3d, y_train)
test_dataset  = FraudDataset(X_test_3d, y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 62
Test batches:  8


In [4]:
class LSTMFraud(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super(LSTMFraud, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]  # lấy output timestep cuối
        return self.classifier(out).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dùng device: {device}")

model = LSTMFraud(input_size=X_train_3d.shape[2]).to(device)
print(model)

Dùng device: cpu
LSTMFraud(
  (lstm): LSTM(192, 64, num_layers=2, batch_first=True, dropout=0.3)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
    (4): Sigmoid()
  )
)


In [5]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 10
train_losses = []

print("Training LSTM...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.4f}")

# Vẽ loss curve
plt.figure(figsize=(8,4))
plt.plot(train_losses, marker='o', color='coral')
plt.title('LSTM Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig('../reports/lstm_loss.png', dpi=80, bbox_inches='tight')
plt.close()
print("Đã lưu loss curve!")

Training LSTM...
Epoch 1/10 — Loss: 0.5957
Epoch 2/10 — Loss: 0.2629
Epoch 3/10 — Loss: 0.1448
Epoch 4/10 — Loss: 0.1119
Epoch 5/10 — Loss: 0.0864
Epoch 6/10 — Loss: 0.0774
Epoch 7/10 — Loss: 0.0628
Epoch 8/10 — Loss: 0.0595
Epoch 9/10 — Loss: 0.0595
Epoch 10/10 — Loss: 0.0478
Đã lưu loss curve!


In [6]:
model.eval()
all_probs = []
all_preds = []

with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        probs = model(X_batch).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)

y_prob_lstm = np.array(all_probs)
y_pred_lstm = np.array(all_preds)

f1    = f1_score(y_test, y_pred_lstm)
mcc   = matthews_corrcoef(y_test, y_pred_lstm)
auprc = average_precision_score(y_test, y_prob_lstm)
auc   = roc_auc_score(y_test, y_prob_lstm)

print(f"\n{'='*40}")
print(f"Model: LSTM")
print(f"{'='*40}")
print(f"F1 Score:  {f1:.4f}")
print(f"MCC:       {mcc:.4f}")
print(f"AUPRC:     {auprc:.4f}")
print(f"ROC-AUC:   {auc:.4f}")
print(f"\n{classification_report(y_test, y_pred_lstm, target_names=['Legit','Fraud'])}")

# Lưu model
torch.save(model.state_dict(), '../models/lstm_fraud.pth')
print("✅ Đã lưu model LSTM!")


Model: LSTM
F1 Score:  0.2388
MCC:       0.2156
AUPRC:     0.2229
ROC-AUC:   0.6694

              precision    recall  f1-score   support

       Legit       0.97      0.98      0.97      1923
       Fraud       0.28      0.21      0.24        77

    accuracy                           0.95      2000
   macro avg       0.62      0.59      0.61      2000
weighted avg       0.94      0.95      0.95      2000

✅ Đã lưu model LSTM!
